In [1]:
from dgl.data import RomanEmpireDataset
dataset = RomanEmpireDataset()
g = dataset[0]
num_classes = dataset.num_classes

/home/nchervov/other_datasets/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Done loading data from cached files.


In [2]:
feat = g.ndata["feat"]

In [3]:
import torch
from torch_geometric.utils import from_dgl

pyg_data = from_dgl(g)

pyg_data.x = g.ndata["feat"]
pyg_data.y = g.ndata["label"]

pyg_data.train_mask = g.ndata["train_mask"][:, 0]
pyg_data.val_mask = g.ndata["val_mask"][:, 0]
pyg_data.test_mask = g.ndata["test_mask"][:, 0]

In [4]:
from torch_geometric.utils import add_self_loops
pyg_data.edge_index, _ = add_self_loops(pyg_data.edge_index, num_nodes=pyg_data.num_nodes)

In [5]:
from torch.nn import functional as F

def train_step(model, optimizer, data):
    model.train()
    optimizer.zero_grad()
    out = model(data, data.x)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    out = model(data, data.x)
    pred = out[mask].argmax(dim=1)
    acc = (pred == data.y[mask]).float().mean().item()
    loss = F.cross_entropy(out[mask], data.y[mask]).item()
    return acc, loss

In [6]:
import torch
import torch.nn.functional as F
from models import Model

device = torch.device("cuda", 1)
data = pyg_data.to(device)

In [7]:
MODEL_TYPE = "GT-sep"
NORMALIZATION = "LayerNorm"

In [8]:
import numpy as np
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


best_val_result = 0
best_layers = None

for num_layers in range(1, 6):
    print(f"Training model with {num_layers} layers")
    model = Model(
        model_name=MODEL_TYPE,
        num_layers=num_layers,
        input_dim=data.x.size(1),
        hidden_dim=512,
        output_dim=num_classes,
        hidden_dim_multiplier=1,
        num_heads=8,
        normalization=NORMALIZATION,
        dropout=0.2,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0)

    num_steps = 1000
    patience = 3
    best_val_acc = 0.0
    best_step = 0
    steps_without_improvement = 0

    for step in range(1, num_steps + 1):
        train_loss = train_step(model, optimizer, data)
        val_acc, val_loss = evaluate(model, data, data.val_mask)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_step = step
            steps_without_improvement = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            steps_without_improvement += 1

        if steps_without_improvement >= patience * 100:
            print(f"Early stopping at step {step} (no improvement for {patience * 100} steps)")
            break

        if step % 100 == 0 or step == 1:
            train_acc, _ = evaluate(model, data, data.train_mask)
            print(f"Step {step:4d} | Train loss {train_loss:.4f} | Train acc {train_acc:.4f} "
                f"| Val loss {val_loss:.4f} | Val acc {val_acc:.4f}")

    print(f"\nBest val acc: {best_val_acc:.4f} at step {best_step}")
    if best_val_acc > best_val_result:
        best_val_result = best_val_acc
        best_layers = num_layers


print(f"Best layers: {best_layers} with val acc {best_val_acc:.4f}")

model = Model(
    model_name=MODEL_TYPE,
    num_layers=best_layers,
    input_dim=data.x.size(1),
    hidden_dim=512,
    output_dim=num_classes,
    hidden_dim_multiplier=1,
    num_heads=8,
    normalization=NORMALIZATION,
    dropout=0.2,
).to(device)

model.load_state_dict(torch.load("best_model.pt", weights_only=True))
test_acc, test_loss = evaluate(model, data, data.test_mask)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Training model with 1 layers


Step    1 | Train loss 3.0594 | Train acc 0.0487 | Val loss 2.8978 | Val acc 0.0558
Step  100 | Train loss 1.0664 | Train acc 0.7036 | Val loss 1.0124 | Val acc 0.6884
Step  200 | Train loss 0.7826 | Train acc 0.7833 | Val loss 0.8200 | Val acc 0.7470
Step  300 | Train loss 0.6275 | Train acc 0.8306 | Val loss 0.7416 | Val acc 0.7666
Step  400 | Train loss 0.5084 | Train acc 0.8732 | Val loss 0.7133 | Val acc 0.7850
Step  500 | Train loss 0.4147 | Train acc 0.9102 | Val loss 0.7207 | Val acc 0.7922
Step  600 | Train loss 0.3352 | Train acc 0.9423 | Val loss 0.7557 | Val acc 0.7933
Step  700 | Train loss 0.2654 | Train acc 0.9631 | Val loss 0.8126 | Val acc 0.7938
Step  800 | Train loss 0.2064 | Train acc 0.9783 | Val loss 0.8696 | Val acc 0.7933
Step  900 | Train loss 0.1669 | Train acc 0.9908 | Val loss 0.9353 | Val acc 0.7919
Step 1000 | Train loss 0.1298 | Train acc 0.9957 | Val loss 0.9938 | Val acc 0.7903

Best val acc: 0.7903 at step 1000
Training model with 2 layers
Step    1 | 

In [8]:
TABM_INITS = 4

In [9]:
from torch.nn import functional as F

def train_step_tabm(model, optimizer, data):
    model.train()
    optimizer.zero_grad()
    for i in range(TABM_INITS):
        out = model(data, data.x, tabm_seed=i)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask]) / TABM_INITS
        loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_tabm(model, data, mask):
    model.eval()
    outs = []
    for i in range(TABM_INITS):
        out = model(data, data.x, tabm_seed=i)
        outs.append(out)
    pred = torch.stack(outs).mean(dim=0)[mask].argmax(dim=1)
    acc = (pred == data.y[mask]).float().mean().item()
    loss = F.cross_entropy(outs[0][mask], data.y[mask]).item()
    return acc, loss

In [ ]:
from models import TABMModel
import numpy as np
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


best_val_result = 0
best_layers = None

for num_layers in [5, 4, 3, 2, 1]:
    print(f"Training model with {num_layers} layers")
    model = TABMModel(
        model_name=MODEL_TYPE,
        num_layers=num_layers,
        input_dim=data.x.size(1),
        hidden_dim=512,
        output_dim=num_classes,
        hidden_dim_multiplier=1,
        num_heads=8,
        normalization=NORMALIZATION,
        dropout=0.2,
        tabm_inits=TABM_INITS,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0)

    num_steps = 1000
    patience = 3
    best_val_acc = 0.0
    best_step = 0
    steps_without_improvement = 0

    for step in range(1, num_steps + 1):
        train_loss = train_step_tabm(model, optimizer, data)
        val_acc, val_loss = evaluate_tabm(model, data, data.val_mask)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_step = step
            steps_without_improvement = 0
            torch.save(model.state_dict(), "best_model_tabm.pt")
        else:
            steps_without_improvement += 1

        if steps_without_improvement >= patience * 100:
            print(f"Early stopping at step {step} (no improvement for {patience * 100} steps)")
            break

        if step % 100 == 0 or step == 1:
            train_acc, _ = evaluate_tabm(model, data, data.train_mask)
            print(f"Step {step:4d} | Train loss {train_loss:.4f} | Train acc {train_acc:.4f} "
                f"| Val loss {val_loss:.4f} | Val acc {val_acc:.4f}")

    print(f"\nBest val acc: {best_val_acc:.4f} at step {best_step}")
    if best_val_acc > best_val_result:
        best_val_result = best_val_acc
        best_layers = num_layers


print(f"Best layers: {best_layers} with val acc {best_val_acc:.4f}")

model = TABMModel(
    model_name=MODEL_TYPE,
    num_layers=best_layers,
    input_dim=data.x.size(1),
    hidden_dim=512,
    output_dim=num_classes,
    hidden_dim_multiplier=1,
    num_heads=8,
    normalization=NORMALIZATION,
    dropout=0.2,
    tabm_inits=TABM_INITS,
).to(device)

model.load_state_dict(torch.load("best_model_tabm.pt", weights_only=True))
test_acc, test_loss = evaluate(model, data, data.test_mask)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Training model with 5 layers


Step    1 | Train loss 0.9268 | Train acc 0.1823 | Val loss 3.5845 | Val acc 0.1795
Step  100 | Train loss 0.2701 | Train acc 0.7231 | Val loss 1.0262 | Val acc 0.7003
Step  200 | Train loss 0.2032 | Train acc 0.7895 | Val loss 0.8262 | Val acc 0.7568
Step  300 | Train loss 0.1621 | Train acc 0.8400 | Val loss 0.6941 | Val acc 0.7929
Step  400 | Train loss 0.1285 | Train acc 0.8836 | Val loss 0.6068 | Val acc 0.8282
Step  500 | Train loss 0.1034 | Train acc 0.9185 | Val loss 0.5486 | Val acc 0.8489
Step  600 | Train loss 0.0829 | Train acc 0.9469 | Val loss 0.5244 | Val acc 0.8635
Step  700 | Train loss 0.0660 | Train acc 0.9686 | Val loss 0.5213 | Val acc 0.8741
Step  800 | Train loss 0.0507 | Train acc 0.9850 | Val loss 0.5275 | Val acc 0.8807
Step  900 | Train loss 0.0394 | Train acc 0.9935 | Val loss 0.5401 | Val acc 0.8849


In [10]:
from models import TABMModel
import numpy as np
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False